# Get YouTube data

Collect video stats from T-pop girl group channels. This data is used as a benchmark for the VELORA launch campaign.

In [1]:
import os
import re
import time
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()
youtube = build("youtube", "v3", developerKey=os.getenv("YOUTUBE_API_KEY"))

## 1. Find the channels

Search each group and print the top 3 channels. Check on YouTube which one is the official channel.

Note: search uses 100 quota units per call (about 800 for all groups), so only run this once.

In [2]:
groups = ["4EVE", "PiXXiE", "Pretzelle", "ALALA", "MXFRUIT", "VIIS", "EMPRESS", "FELIZZ"]

for group in groups:
    res = youtube.search().list(
        part="snippet",
        q=group + " official",
        type="channel",
        maxResults=3,
    ).execute()

    print("==", group)
    for item in res["items"]:
        print("  ", item["snippet"]["title"], "|", item["snippet"]["channelId"])

== 4EVE
   4EVE | UC9mrIKAyV8AushabYK8i2ew
   4EVE - Topic | UCdmScpl5nFGLy3xCnfHjDsw
   4eve Official | UCcWoETl4rjusqelo15bc27w
== PiXXiE
   PiXXiE | UC0yQ9AFdAp-18aEJWtXFY0Q
   PIX!E Official PH | UC_pqMS2Yp5VFkAYFDvrTmpw
   Pixie Gaming Official | UCDRmsu4J10_n1ctwhUqvsiw
== Pretzelle
   Pretzelle | UCUic_1S8_5yGh-E34h9B2wg
   PRETZELLE - Topic | UCeo6TZ6uFcyw73zBF49uryA
   Pretzzzelle | UClEPypCpt4hTR9plEKajxMw
== ALALA
   ALALA WhiteFox | UCqUqvOZzfOatTkTh8ZrF7uw
   ALALA - Topic | UC4BevqFe8lim_zqUD3uJL4g
   Alala Official | UCHJx9jVwJ8An7PVPUnzHWyg
== MXFRUIT
   KILLPER | UCCdzyRRY1JBPuUPaoO8Q94Q
   Aprettygirls | UCvHezLFbI-VtZ8uJG3b3pig
   JMM jame | UCK8Jv4LGbkLoOrAzdryvaaA
== VIIS
   VIIS Official | UCuBBlYiHdHGPs7mpHhvWzpw
   VIIS - Topic | UC5EryiGHl3EukA4FKOfhafA
   VIIS | UCW0iUX99bKJoTYWLFyEETKQ
== EMPRESS
   EMPRESS - Topic | UCzqHKhScR2G-JEJ8gR0pjQA
   EMPRESS | UCrLP_3RqrrOTG7iVeinjSGg
   Simply Empress | UCNYGDqy0o9kvF08xvGqkS9A
== FELIZZ
   FELIZZ | UCcrw53OZg7Lr2

In [8]:
candidates = {
    "4EVE": ["UC9mrIKAyV8AushabYK8i2ew", "UCcWoETl4rjusqelo15bc27w"],
    "ALALA": ["UCqUqvOZzfOatTkTh8ZrF7uw", "UCHJx9jVwJ8An7PVPUnzHWyg"],
    "VIIS": ["UCuBBlYiHdHGPs7mpHhvWzpw", "UCW0iUX99bKJoTYWLFyEETKQ"],
}

all_ids = []
for ids in candidates.values():
    all_ids.extend(ids)

res = youtube.channels().list(
    part="snippet,statistics",
    id=",".join(all_ids),
).execute()

for item in res["items"]:
    print(
        item["snippet"]["title"], "|",
        item["id"], "|",
        "subs:", item["statistics"].get("subscriberCount"), "|",
        "videos:", item["statistics"].get("videoCount"),
    )

Alala Official | UCHJx9jVwJ8An7PVPUnzHWyg | subs: 4 | videos: 1
4eve Official | UCcWoETl4rjusqelo15bc27w | subs: 130000 | videos: 2703
VIIS Official | UCuBBlYiHdHGPs7mpHhvWzpw | subs: 48900 | videos: 754
VIIS | UCW0iUX99bKJoTYWLFyEETKQ | subs: 0 | videos: 3
ALALA WhiteFox | UCqUqvOZzfOatTkTh8ZrF7uw | subs: 17000 | videos: 614
4EVE | UC9mrIKAyV8AushabYK8i2ew | subs: 1370000 | videos: 2527


In [9]:
res = youtube.search().list(
    part="snippet",
    q="MXFRUIT Strawberry Ice Cream MV",
    type="video",
    maxResults=5,
).execute()

for item in res["items"]:
    print(item["snippet"]["title"], "|", item["snippet"]["channelTitle"], "|", item["snippet"]["channelId"])

MXFRUIT &#39;strawberry ice cream (สตรอว์เบอร์รีไอศกรีม)&#39; OFFICIAL MV | TADA LABELS | UChvk1ez0-qeKEn12-CYxrzQ
MXFRUIT - Strawberry Ice cream [Color Coded Lyrics] | CC THAI SUB | MASTERSTUDIOZ | UCXZ-5qrIaeOljBUkuiashmg
MXFRUIT &#39;strawberry ice cream (สตรอว์เบอร์รีไอศกรีม)&#39; Choreography VDO | TADA LABELS | UChvk1ez0-qeKEn12-CYxrzQ
[ENGLISH] #MXFRUIT Thank for listening and watching Music Video strawberry ice cream | MXFRUIT | UCpi9qhFqWeV-TrWBzZUyENg
[SPAIN] #MXFRUIT Thank for listening and watching Music Video strawberry ice cream | MXFRUIT | UCpi9qhFqWeV-TrWBzZUyENg


Copy the correct channel id for each group here. Leave it empty if you can't find one.

In [11]:
channel_ids = {
    "4EVE": "UC9mrIKAyV8AushabYK8i2ew",
    "PiXXiE": "UC0yQ9AFdAp-18aEJWtXFY0Q",
    "Pretzelle": "UCUic_1S8_5yGh-E34h9B2wg",
    "ALALA": "UCqUqvOZzfOatTkTh8ZrF7uw",
    "MXFRUIT": "UChvk1ez0-qeKEn12-CYxrzQ",
    "VIIS": "UCuBBlYiHdHGPs7mpHhvWzpw",
    "EMPRESS": "UCrLP_3RqrrOTG7iVeinjSGg",
    "FELIZZ": "UCcrw53OZg7Lr2GwoNXBtEJg",
}

## 2. Channel info

In [12]:
channel_rows = []

for group, channel_id in channel_ids.items():
    if channel_id == "":
        print("skip", group)
        continue

    res = youtube.channels().list(
        part="snippet,statistics,contentDetails",
        id=channel_id,
    ).execute()
    item = res["items"][0]

    channel_rows.append({
        "group": group,
        "channel_id": channel_id,
        "channel_name": item["snippet"]["title"],
        "subscribers": int(item["statistics"].get("subscriberCount", 0)),
        "total_views": int(item["statistics"].get("viewCount", 0)),
        "video_count": int(item["statistics"].get("videoCount", 0)),
        "uploads_playlist": item["contentDetails"]["relatedPlaylists"]["uploads"],
    })

channels = pd.DataFrame(channel_rows)
channels

,group,channel_id,channel_name,subscribers,total_views,video_count,uploads_playlist
0,4EVE,UC9mrIKAyV8AushabYK8i2ew,4EVE,1370000,1002205729,2527,UU9mrIKAyV8AushabYK8i2ew
1,PiXXiE,UC0yQ9AFdAp-18aEJWtXFY0Q,PiXXiE,223000,79010391,2066,UU0yQ9AFdAp-18aEJWtXFY0Q
2,Pretzelle,UCUic_1S8_5yGh-E34h9B2wg,Pretzelle,156000,122052352,520,UUUic_1S8_5yGh-E34h9B2wg
3,ALALA,UCqUqvOZzfOatTkTh8ZrF7uw,ALALA WhiteFox,17000,6467142,614,UUqUqvOZzfOatTkTh8ZrF7uw
4,MXFRUIT,UChvk1ez0-qeKEn12-CYxrzQ,TADA LABELS,665000,309152584,552,UUhvk1ez0-qeKEn12-CYxrzQ
5,VIIS,UCuBBlYiHdHGPs7mpHhvWzpw,VIIS Official,48900,14068262,754,UUuBBlYiHdHGPs7mpHhvWzpw
6,EMPRESS,UCrLP_3RqrrOTG7iVeinjSGg,EMPRESS,25700,7611409,543,UUrLP_3RqrrOTG7iVeinjSGg
7,FELIZZ,UCcrw53OZg7Lr2GwoNXBtEJg,FELIZZ,117000,39802456,961,UUcrw53OZg7Lr2GwoNXBtEJg


## 3. Get all videos from each channel

In [13]:
def get_video_ids(playlist_id):
    video_ids = []
    next_page = None

    while True:
        res = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page,
        ).execute()

        for item in res["items"]:
            video_ids.append(item["contentDetails"]["videoId"])

        next_page = res.get("nextPageToken")
        if next_page is None:
            break

    return video_ids


def duration_to_seconds(duration):
    # YouTube gives duration like PT3M45S
    total = 0
    hours = re.search(r"(\d+)H", duration)
    minutes = re.search(r"(\d+)M", duration)
    seconds = re.search(r"(\d+)S", duration)
    if hours:
        total += int(hours.group(1)) * 3600
    if minutes:
        total += int(minutes.group(1)) * 60
    if seconds:
        total += int(seconds.group(1))
    return total


def get_video_details(video_ids):
    rows = []

    # the API takes max 50 ids per call
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i + 50]
        res = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(batch),
        ).execute()

        for item in res["items"]:
            stats = item["statistics"]
            rows.append({
                "video_id": item["id"],
                "title": item["snippet"]["title"],
                "published_at": item["snippet"]["publishedAt"],
                "duration_sec": duration_to_seconds(item["contentDetails"]["duration"]),
                "views": int(stats.get("viewCount", 0)),
                "likes": int(stats.get("likeCount", 0)),
                "comments": int(stats.get("commentCount", 0)),
            })

    return rows

In [14]:
all_videos = []

for _, row in channels.iterrows():
    print("getting", row["group"])
    ids = get_video_ids(row["uploads_playlist"])
    videos = get_video_details(ids)

    for v in videos:
        v["group"] = row["group"]

    all_videos.extend(videos)
    time.sleep(1)

df = pd.DataFrame(all_videos)
print(len(df), "videos")
df.head()

getting 4EVE
getting PiXXiE
getting Pretzelle
getting ALALA
getting MXFRUIT
getting VIIS
getting EMPRESS
getting FELIZZ
8533 videos


,video_id,title,published_at,duration_sec,views,likes,comments,group
0,n96tlHlCoRs,คุณคิดกับฉันมากกว่า..... 👉🏻👈🏻 #4EVE #XOXOenter...,2026-11-09T10:00:38Z,31,11659,990,22,4EVE
1,HPk2Y9G_o_k,เกือบแพ้เสียงในหัว! เวรี่คิวต์จริงๆ 😤4EVE ‘ที่...,2026-11-09T08:30:25Z,73,5779,582,8,4EVE
2,F50wpt5zyVo,ไอต้าวเด้งดึ๊ง👀🤘🏻‧₊˚⋅ คอนเสิร์ตการกุศลรอบพิเศษ...,2026-11-09T08:00:22Z,11,17420,759,6,4EVE
3,IsACCWLmTso,แต่วันนี้ฉันขยี้ดราม่าทิ้ง💃🏻💥 ‧₊˚⋅ คอนเสิร์ตกา...,2026-11-09T07:30:01Z,80,2698,288,2,4EVE
4,dhsunJVlPWI,ไปล้อเล่นไกลๆเลยนะ!!! ‧₊˚⋅ คอนเสิร์ตการกุศลรอบ...,2026-11-09T07:15:21Z,84,1903,263,5,4EVE


In [19]:
df["collected_at"] = pd.Timestamp("2026-11-10", tz="Asia/Bangkok")

channels.to_csv("../data/raw/channels.csv", index=False)
df.to_csv("../data/raw/videos.csv", index=False)
print("saved", len(df), "videos")

saved 8079 videos


In [20]:
df.groupby("group")[["views", "likes", "comments"]].sum()

,views,likes,comments
group,,,
4EVE,905964033,14275060,410543
ALALA,3538577,114354,2429
EMPRESS,5681762,180613,5280
FELIZZ,36185958,1761440,22188
MXFRUIT,13091689,321385,14731
PiXXiE,74088898,3240999,34716
Pretzelle,105430897,1130043,52824
VIIS,12400964,260427,4813


## 4. Clean up

- Change the time to Thai time
- Mark short videos. Shorts can be up to 3 minutes, so this is only a rough guess
- Engagement rate = (likes + comments) / views

In [15]:
df["published_at"] = pd.to_datetime(df["published_at"]).dt.tz_convert("Asia/Bangkok")
df["is_short"] = df["duration_sec"] <= 180
df["engagement_rate"] = (df["likes"] + df["comments"]) / df["views"].where(df["views"] > 0)

df.groupby("group")[["views", "likes", "comments"]].sum()

,views,likes,comments
group,,,
4EVE,905964033,14275060,410543
ALALA,3538577,114354,2429
EMPRESS,5681762,180613,5280
FELIZZ,36185958,1761440,22188
MXFRUIT,309274635,5200367,276551
PiXXiE,74088898,3240999,34716
Pretzelle,105430897,1130043,52824
VIIS,12400964,260427,4813


If a group posts on its label's channel (not its own), the channel will have videos from other artists too.
Put that group in the list below to keep only videos with the group name in the title.

In [16]:
label_channel_groups = ["MXFRUIT"]

for group in label_channel_groups:
    not_this_group = (df["group"] == group) & (~df["title"].str.contains(group, case=False))
    df = df[~not_this_group]

df["group"].value_counts()

group
4EVE         2528
PiXXiE       2059
FELIZZ        961
VIIS          754
ALALA         614
EMPRESS       543
Pretzelle     521
MXFRUIT        99
Name: count, dtype: int64

## 5. Save

In [17]:
channels.to_csv("../data/raw/channels.csv", index=False)
df.to_csv("../data/raw/videos.csv", index=False)
print("saved")

saved
